
# 01 · Limpieza y EDA — Market Data España (ADR, Ocupación, RevPAR)

Este notebook carga el dataset base (`hotel_bookings.csv`), lo limpia y normaliza, 
crea variables derivadas, calcula métricas mensuales (ADR realizado, % cancelación, revenue, noches reservadas) 
y exporta datasets listos para Power BI en `data/processed/`.


In [ ]:


import os 
from datetime import datetime 
import pandas as pd 
import numpy as np 

pd .set_option ("display.max_columns",120 )
pd .set_option ("display.width",140 )


CWD =os .getcwd ()
PROJ =os .path .abspath (os .path .join (CWD ,".."))if os .path .basename (CWD ).lower ()=="notebooks"else CWD 


DATA_RAW =os .path .join (PROJ ,"data","raw")
DATA_PROC =os .path .join (PROJ ,"data","processed")
DOCS_DIR =os .path .join (PROJ ,"docs")

os .makedirs (DATA_RAW ,exist_ok =True )
os .makedirs (DATA_PROC ,exist_ok =True )
os .makedirs (DOCS_DIR ,exist_ok =True )

PROJ ,DATA_RAW ,DATA_PROC 


In [ ]:


CATALOG_CSV =os .path .join (DOCS_DIR ,"data_catalog.csv")

def register_dataset (name :str ,source :str ,file_path :str ,notes :str =""):
    row ={
    "registered_at":datetime .now ().isoformat (timespec ="seconds"),
    "name":name ,
    "source":source ,
    "file_path":os .path .relpath (file_path ,PROJ ),
    "notes":notes 
    }
    if os .path .exists (CATALOG_CSV ):
        df =pd .read_csv (CATALOG_CSV )
        df =pd .concat ([df ,pd .DataFrame ([row ])],ignore_index =True )
    else :
        df =pd .DataFrame ([row ])
    df .to_csv (CATALOG_CSV ,index =False )
    return df .tail (5 )

def quick_overview (df :pd .DataFrame ,top_missing :int =15 ):
    info =pd .DataFrame ({
    "dtype":df .dtypes .astype (str ),
    "n_unique":df .nunique (),
    "n_missing":df .isna ().sum ()
    })
    info ["missing_pct"]=(info ["n_missing"]/len (df )*100 ).round (2 )
    display (info .sort_values ("missing_pct",ascending =False ).head (top_missing ))
    return info 


In [ ]:


hotel_csv =os .path .join (DATA_RAW ,"hotel_bookings.csv")
assert os .path .exists (hotel_csv ),f"No encuentro {hotel_csv}. Coloca el CSV en data/raw/ y vuelve a ejecutar."

hotel =pd .read_csv (hotel_csv )
register_dataset (
name ="hotel_bookings",
source ="Kaggle: jessemostipak/hotel-booking-demand",
file_path =hotel_csv ,
notes ="Original (raw)"
)

hotel .shape ,hotel .head (3 )


In [ ]:


hotel .info ()
_ =quick_overview (hotel ,top_missing =20 )


In [ ]:


df =hotel .copy ()

month_map ={m :i for i ,m in enumerate (
["January","February","March","April","May","June","July","August","September","October","November","December"],start =1 )}
df ["arrival_month_num"]=df ["arrival_date_month"].map (month_map )
df ["arrival_date"]=pd .to_datetime (
dict (year =df ["arrival_date_year"],
month =df ["arrival_month_num"],
day =df ["arrival_date_day_of_month"]),
errors ="coerce"
)

for c in ["lead_time","stays_in_weekend_nights","stays_in_week_nights","adults","children","babies","adr"]:
    df [c ]=pd .to_numeric (df [c ],errors ="coerce")

df [["children","babies"]]=df [["children","babies"]].fillna (0 )
df ["stay_nights"]=(df ["stays_in_weekend_nights"].fillna (0 )+df ["stays_in_week_nights"].fillna (0 )).astype (float )
df ["guests"]=(df ["adults"].fillna (0 )+df ["children"].fillna (0 )+df ["babies"].fillna (0 )).astype (float )

df ["adr"]=df ["adr"].clip (lower =0 )
df ["stay_nights"]=df ["stay_nights"].clip (lower =0 )
df ["revenue_reservation"]=df ["adr"]*df ["stay_nights"]

df ["is_canceled"]=df ["is_canceled"].fillna (0 ).astype (int )
df ["country"]=df ["country"].fillna ("Unknown")
df ["market_segment"]=df ["market_segment"].fillna ("Unknown")

dup_before =df .duplicated ().sum ()
df =df .drop_duplicates ().reset_index (drop =True )
dup_after =df .duplicated ().sum ()

{"duplicados_antes":dup_before ,"duplicados_despues":dup_after }


In [ ]:


nulls_top =df .isna ().mean ().sort_values (ascending =False ).head (10 )
summary ={
"min_date":df ["arrival_date"].min (),
"max_date":df ["arrival_date"].max (),
"reservas":len (df ),
"noches_media":df ["stay_nights"].mean (),
"adr_medio":df ["adr"].mean (),
"revenue_medio_reserva":df ["revenue_reservation"].mean (),
}
nulls_top ,summary 


In [ ]:


cols_bi =[
"arrival_date","hotel","is_canceled","lead_time","stay_nights","guests","adr",
"revenue_reservation","country","market_segment","reserved_room_type","assigned_room_type"
]
df_bi =df [cols_bi ].copy ()

bi_csv =os .path .join (DATA_PROC ,"hotel_bookings_clean_for_bi.csv")
df_bi .to_csv (bi_csv ,index =False )

register_dataset (
name ="hotel_bookings_clean_for_bi",
source ="Transformación local desde hotel_bookings (raw)",
file_path =bi_csv ,
notes ="Dataset limpio/base para Power BI"
)

bi_csv 


In [ ]:


def monthly_metrics (df_in :pd .DataFrame ):
    tmp =df_in .copy ()
    tmp ["month"]=tmp ["arrival_date"].dt .to_period ("M").dt .to_timestamp ()

    agg_all =(
    tmp .groupby ("month")
    .agg (reservas =("is_canceled","size"),
    canceladas =("is_canceled","sum"))
    .assign (cancel_rate_pct =lambda d :(d ["canceladas"]/d ["reservas"]*100 ).round (2 ))
    )

    not_canceled =tmp [tmp ["is_canceled"]==0 ]
    agg_rev =(
    not_canceled .groupby ("month")
    .agg (noches_reservadas =("stay_nights","sum"),
    revenue_total =("revenue_reservation","sum"))
    .assign (adr_realizado =lambda d :np .where (d ["noches_reservadas"]>0 ,
    d ["revenue_total"]/d ["noches_reservadas"],np .nan ))
    )

    out =agg_all .join (agg_rev ,how ="left")
    return out .reset_index ()

m_all =monthly_metrics (df_bi )
m_esp =monthly_metrics (df_bi [df_bi ["country"]=="ESP"])

m_all .head (),m_esp .head ()


In [ ]:


def add_occ_revpar (metrics_df :pd .DataFrame ,available_nights_per_month :dict ):
    dfm =metrics_df .copy ()
    dfm ["available_nights"]=dfm ["month"].map (available_nights_per_month ).astype ("float")
    dfm ["occupancy_pct"]=(dfm ["noches_reservadas"]/dfm ["available_nights"]*100 ).round (2 )
    dfm ["revpar"]=(dfm ["revenue_total"]/dfm ["available_nights"]).round (2 )
    return dfm 






In [ ]:


metrics_all_csv =os .path .join (DATA_PROC ,"metrics_monthly_all.csv")
metrics_esp_csv =os .path .join (DATA_PROC ,"metrics_monthly_ESP.csv")

m_all .to_csv (metrics_all_csv ,index =False )
m_esp .to_csv (metrics_esp_csv ,index =False )

register_dataset (
name ="metrics_monthly_all",
source ="Aggregations from hotel_bookings_clean_for_bi",
file_path =metrics_all_csv ,
notes ="ADR realizado, % cancelación, noches reservadas y revenue por mes"
)

register_dataset (
name ="metrics_monthly_ESP",
source ="Aggregations from hotel_bookings_clean_for_bi (country=ESP)",
file_path =metrics_esp_csv ,
notes ="ADR realizado, % cancelación, noches reservadas y revenue por mes (origen España)"
)

metrics_all_csv ,metrics_esp_csv 
